# Movie Review Sentiment Analysis (spaCy + TF-IDF + Naive Bayes)

In [8]:
import pandas as pd
import spacy
import pickle

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [9]:
df = pd.read_csv(r"C:\Users\sachin-selvam\Documents\movie review\IMDB_Dataset_CLEANED.csv")

df = df.sample(n=1000, random_state=42)

print("Dataset Shape:", df.shape)
print(df["sentiment"].value_counts())

Dataset Shape: (1000, 2)
sentiment
negative    505
positive    495
Name: count, dtype: int64


In [10]:
nlp = spacy.load(
    "en_core_web_sm",
    disable=["parser", "ner"]
)

print("spaCy Model Loaded")

spaCy Model Loaded


In [11]:
clean_reviews = []

for doc in nlp.pipe(
    df["review"].astype(str),
    batch_size=500
):
    tokens = [
        token.lemma_
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and not token.is_space
    ]
    clean_reviews.append(" ".join(tokens))

df["clean_review"] = clean_reviews

print("Preprocessing Completed")

Preprocessing Completed


In [12]:
print(df[["review","clean_review"]].head())

                                                  review  \
4057   First of all, the genre of this movie isn't co...   
31167  Many people have commented that this movie was...   
22806  Well, for a start, I must say that, here, in R...   
14572  Plot: an amorous couple decide to engage in so...   
11478  I watched this film with a group of Nazis, a F...   

                                            clean_review  
4057   genre movie comedy drama.<br /><br />I low exp...  
31167  people comment movie near good maybe child rea...  
22806  start Russia saga Geralt Rivia know love Andrz...  
14572  plot amorous couple decide engage extra marita...  
11478  watch film group Nazis french Archaeologist ex...  


In [13]:
tfidf = TfidfVectorizer(max_features=5000)

X = tfidf.fit_transform(df["clean_review"])
y = df["sentiment"]

print("TF-IDF Completed")

TF-IDF Completed


In [14]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Train Shape:", X_train.shape)
print("Test Shape:", X_test.shape)

Train Shape: (800, 5000)
Test Shape: (200, 5000)


In [15]:
model = MultinomialNB()
model.fit(X_train, y_train)

print("Model Training Completed")

Model Training Completed


In [16]:
y_pred = model.predict(X_test)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", round(accuracy * 100, 2), "%")

Accuracy: 81.0 %


In [17]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

    negative       0.78      0.87      0.82       101
    positive       0.85      0.75      0.80        99

    accuracy                           0.81       200
   macro avg       0.81      0.81      0.81       200
weighted avg       0.81      0.81      0.81       200



In [18]:
print(confusion_matrix(y_test, y_pred))

[[88 13]
 [25 74]]


In [19]:
pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(tfidf, open("tfidf.pkl", "wb"))

print("Model Saved Successfully")

Model Saved Successfully


In [20]:
def predict_sentiment(review):

    doc = nlp(review.lower())

    tokens = [
        token.lemma_
        for token in doc
        if not token.is_stop
        and not token.is_punct
        and not token.is_space
    ]

    clean_review = " ".join(tokens)

    review_vector = tfidf.transform([clean_review])

    prediction = model.predict(review_vector)

    return prediction[0]

In [21]:
review = input("Enter Movie Review: ")
result = predict_sentiment(review)

print("Predicted Sentiment:", result)

Predicted Sentiment: negative
